In [9]:
import sys
import subprocess
import importlib

def ensure_packages(packages):
    missing = []
    for pkg in packages:
        try:
            importlib.import_module(pkg)
        except Exception:
            missing.append(pkg)
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    else:
        print("All packages already installed:", packages)

ensure_packages(["selenium", "webdriver_manager", "beautifulsoup4", "pandas"])

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
import time, re
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException


Installing missing packages: ['beautifulsoup4']


In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time


def scrape_imdb_reviews(product_id, headless=False, delay=0.3, timeout=2):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120 Safari/537.36"
    )

    driver = webdriver.Chrome(options=opts)
    driver.set_window_size(1200, 900)
    url = f"https://www.imdb.com/title/{product_id}/reviews"
    print("Opening:", url)
    driver.get(url)
    time.sleep(2)

    # Keep clicking "Load More" until it's gone
    while True:
        try:
            load_more = WebDriverWait(driver, timeout).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "span.ipc-see-more__text"))
            )
            driver.execute_script("arguments[0].click();", load_more)
            print("Clicked Load More...")
            time.sleep(delay)
        except TimeoutException:
            print("No more 'Load More' button.")
            break

    # Click all "See more" buttons inside reviews
    see_more_buttons = driver.find_elements(By.CSS_SELECTOR, "button.ipc-overflowText-overlay")
    print(f"Expanding {len(see_more_buttons)} 'See more' buttons...")
    for btn in see_more_buttons:
        try:
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(0.1)
        except:
            pass

    wrappers = driver.find_elements(By.CSS_SELECTOR, "article.user-review-item")
    print("Total reviews found:", len(wrappers))

    reviews_data = []
    for w in wrappers:
        # Full review text
        try:
            text = w.find_element(
                By.CSS_SELECTOR, "div.ipc-overflowText--children div.ipc-html-content"
            ).text.strip()
        except:
            try:
                text = w.find_element(
                    By.CSS_SELECTOR, "div[data-testid='review-overflow']"
                ).text.strip()
            except:
                text = ""

        # Rating
        try:
            rating = w.find_element(
                By.CSS_SELECTOR, "span.ipc-rating-star--rating"
            ).text.strip()
        except:
            rating = ""

        # Likes
        try:
            likes = w.find_element(
                By.CSS_SELECTOR, "span.ipc-voting__label__count.ipc-voting__label__count--up"
            ).text.strip()
        except:
            likes = ""

        # Dislikes
        try:
            dislikes = w.find_element(
                By.CSS_SELECTOR, "span.ipc-voting__label__count.ipc-voting__label__count--down"
            ).text.strip()
        except:
            dislikes = ""

        reviews_data.append({
            "rating": rating,
            "review": text,
            "likes": likes,
            "dislikes": dislikes
        })

    driver.quit()
    return reviews_data


In [13]:

def url_link(product_url, save_csv=True):
    s = product_url.strip("/").split("/")
    product_id = s[-1]   # extract product ID
    reviews = scrape_imdb_reviews(product_id)  
    df = pd.DataFrame(reviews)
    if save_csv:
        df.to_csv(f"imdb_reviews_{product_id}.csv", index=False, encoding="utf-8")
        print(f"Saved reviews to imdb_reviews_{product_id}.csv")
    return df


In [ ]:
df=url_link("https://www.imdb.com/title/tt0468569/")
df

Opening: https://www.imdb.com/title/tt0468569/reviews
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
Clicked Load More...
No more 'Load More' button.
Expanding 86 'See more' buttons...


In [1]:
inp=url_link(input("enter any  movie link of imdb"))
inp

NameError: name 'url_link' is not defined